# Experiment: PhoWhisper.cpp Colab Pro Canonical Run

Objective:
- Convert PhoWhisper into a whisper.cpp runtime artifact on Colab Pro.
- Benchmark the converted model against Whisper baselines on a Vietnamese test set.
- Produce paper-ready evidence: figures, qualitative failure cases, manifests, and a downloadable results bundle.

Outputs:
- Converted `PhoWhisper.cpp` artifacts (`fp16`, optional `q4_0`, optional `q5_0`).
- Benchmark JSON/CSV/Markdown reports.
- Figures and qualitative evidence under `evidence/`.
- One ZIP bundle you can download back to local storage.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

PROJECT_ROOT = Path('/content/phowhisper_cpp_colab')
WORKSPACE_ROOT = PROJECT_ROOT / 'workspace'
REPO_ARCHIVE_URL = ''  # Optional: direct ZIP URL of this repo or a prepared bundle.
BUNDLE_ZIP_URL = ''    # Optional: direct ZIP URL from `package-colab`.
DATASET_ARCHIVE_URL = ''  # Prefer a direct http(s) link or gdrive://FILE_ID.
DATASET_ARCHIVE_PATH = '' # Optional mounted Drive path, e.g. /content/drive/MyDrive/dataset.zip
HF_MODEL_REPO = 'vinai/PhoWhisper-large'
LANGUAGE = 'vi'
THREADS = min(8, max(2, (os.cpu_count() or 4) - 1))
MAX_FILES = None  # Set to a small int for a quick sanity run.
RUN_Q4 = True
RUN_Q5 = True
RUN_BASELINE_LARGE_V2 = True
RUN_BASELINE_LARGE_V3 = True
QUALITATIVE_TOP_N = 3
BENCHMARK_RUN_ID = 'run_colab_canonical'
PROJECT_ROOT


In [ ]:
import shutil
import subprocess
import sys
import urllib.request
import zipfile

def run(command: list[str], cwd: Path | None = None) -> None:
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, check=True, cwd=str(cwd) if cwd else None)

def download(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, destination.open('wb') as handle:
        shutil.copyfileobj(response, handle, length=1024 * 1024)
    return destination

def find_source_root(base: Path) -> Path:
    candidates = [base, *[path for path in base.glob('*') if path.is_dir()]]
    for candidate in candidates:
        if (candidate / 'scripts' / 'phowhisper_cpp_experiment.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate scripts/phowhisper_cpp_experiment.py after extraction.')

run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'git', 'cmake', 'build-essential', 'ffmpeg'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'transformers', 'safetensors', 'matplotlib', 'numpy', 'torch', 'gdown'])

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

archive_target = None
if REPO_ARCHIVE_URL:
    archive_target = download(REPO_ARCHIVE_URL, PROJECT_ROOT / 'repo_bundle.zip')
elif BUNDLE_ZIP_URL:
    archive_target = download(BUNDLE_ZIP_URL, PROJECT_ROOT / 'repo_bundle.zip')

if archive_target is not None:
    with zipfile.ZipFile(archive_target, 'r') as archive:
        archive.extractall(PROJECT_ROOT)

SOURCE_ROOT = find_source_root(PROJECT_ROOT)
SCRIPT_PATH = SOURCE_ROOT / 'scripts' / 'phowhisper_cpp_experiment.py'
print('SOURCE_ROOT =', SOURCE_ROOT)
print('SCRIPT_PATH =', SCRIPT_PATH)


In [ ]:
from huggingface_hub import snapshot_download

run([sys.executable, str(SCRIPT_PATH), 'init-workspace', '--workspace-root', str(WORKSPACE_ROOT)])

if DATASET_ARCHIVE_URL:
    run([
        sys.executable, str(SCRIPT_PATH), 'fetch-dataset',
        '--workspace-root', str(WORKSPACE_ROOT),
        '--dataset-id', 'benchmark_vi_longform_v1',
        '--source-url', DATASET_ARCHIVE_URL,
        '--force',
    ])
elif DATASET_ARCHIVE_PATH:
    run([
        sys.executable, str(SCRIPT_PATH), 'fetch-dataset',
        '--workspace-root', str(WORKSPACE_ROOT),
        '--dataset-id', 'benchmark_vi_longform_v1',
        '--archive-path', DATASET_ARCHIVE_PATH,
        '--force',
    ])
else:
    print('No dataset source provided. Set DATASET_ARCHIVE_URL or DATASET_ARCHIVE_PATH before running this cell.')

hf_model_dir = WORKSPACE_ROOT / 'external' / 'models' / 'hf' / 'phowhisper-large'
snapshot_download(repo_id=HF_MODEL_REPO, local_dir=str(hf_model_dir))

whisper_cpp_dir = WORKSPACE_ROOT / 'external' / 'vendor' / 'whisper.cpp'
openai_whisper_dir = WORKSPACE_ROOT / 'external' / 'vendor' / 'openai-whisper'
if not whisper_cpp_dir.exists():
    run(['git', 'clone', 'https://github.com/ggml-org/whisper.cpp', str(whisper_cpp_dir)])
if not openai_whisper_dir.exists():
    run(['git', 'clone', 'https://github.com/openai/whisper', str(openai_whisper_dir)])

run(['cmake', '-S', str(whisper_cpp_dir), '-B', str(whisper_cpp_dir / 'build')])
run(['cmake', '--build', str(whisper_cpp_dir / 'build'), '-j'])

download_script = whisper_cpp_dir / 'models' / 'download-ggml-model.sh'
ggml_dir = WORKSPACE_ROOT / 'external' / 'models' / 'ggml'
ggml_dir.mkdir(parents=True, exist_ok=True)
if RUN_BASELINE_LARGE_V2:
    run(['bash', str(download_script), 'large-v2', str(ggml_dir)])
if RUN_BASELINE_LARGE_V3:
    run(['bash', str(download_script), 'large-v3', str(ggml_dir)])


In [ ]:
run([
    sys.executable, str(SCRIPT_PATH), 'convert',
    '--workspace-root', str(WORKSPACE_ROOT),
    '--whisper-cpp-dir', str(WORKSPACE_ROOT / 'external' / 'vendor' / 'whisper.cpp'),
    '--openai-whisper-dir', str(WORKSPACE_ROOT / 'external' / 'vendor' / 'openai-whisper'),
    '--hf-model-dir', str(WORKSPACE_ROOT / 'external' / 'models' / 'hf' / 'phowhisper-large'),
] + ([] if RUN_Q4 else ['--skip-q4']) + ([] if RUN_Q5 else ['--skip-q5']))

conversion_root = WORKSPACE_ROOT / 'artifacts' / 'conversion'
conversion_dirs = sorted([path for path in conversion_root.iterdir() if path.is_dir()])
if not conversion_dirs:
    raise RuntimeError('No conversion output directory found.')
CONVERSION_DIR = conversion_dirs[-1]
MODELS_DIR = CONVERSION_DIR / 'models'

pho_models = [MODELS_DIR / 'ggml-phowhisper-large.bin']
if RUN_Q4:
    pho_models.append(MODELS_DIR / 'ggml-phowhisper-large-q4_0.bin')
if RUN_Q5:
    pho_models.append(MODELS_DIR / 'ggml-phowhisper-large-q5_0.bin')

for path in pho_models:
    print(path, path.exists())

CONVERSION_DIR


In [ ]:
dataset_dir = WORKSPACE_ROOT / 'data' / 'datasets' / 'benchmark_vi_longform_v1'
output_dir = WORKSPACE_ROOT / 'artifacts' / 'benchmarks' / 'whispercpp' / 'q5_40_long' / BENCHMARK_RUN_ID
whisper_cli = WORKSPACE_ROOT / 'external' / 'vendor' / 'whisper.cpp' / 'build' / 'bin' / 'whisper-cli'
if not whisper_cli.exists():
    whisper_cli = WORKSPACE_ROOT / 'external' / 'vendor' / 'whisper.cpp' / 'build' / 'bin' / 'whisper-cli.exe'

model_specs = [f'pho_large_fp16={MODELS_DIR / "ggml-phowhisper-large.bin"}']
if RUN_Q4:
    model_specs.append(f'pho_large_q4={MODELS_DIR / "ggml-phowhisper-large-q4_0.bin"}')
if RUN_Q5:
    model_specs.append(f'pho_large_q5={MODELS_DIR / "ggml-phowhisper-large-q5_0.bin"}')
if RUN_BASELINE_LARGE_V2:
    model_specs.append(f'whisper_large_v2_q5={WORKSPACE_ROOT / "external" / "models" / "ggml" / "ggml-large-v2-q5_0.bin"}')
if RUN_BASELINE_LARGE_V3:
    model_specs.append(f'whisper_large_v3_q5={WORKSPACE_ROOT / "external" / "models" / "ggml" / "ggml-large-v3-q5_0.bin"}')

benchmark_command = [
    sys.executable, str(SCRIPT_PATH), 'benchmark',
    '--dataset-dir', str(dataset_dir),
    '--output-dir', str(output_dir),
    '--whisper-cli', str(whisper_cli),
    '--language', LANGUAGE,
    '--threads', str(THREADS),
]
if MAX_FILES is not None:
    benchmark_command.extend(['--max-files', str(MAX_FILES)])
for spec in model_specs:
    benchmark_command.extend(['--model', spec])
run(benchmark_command)
run([
    sys.executable, str(SCRIPT_PATH), 'render-evidence',
    '--benchmark-dir', str(output_dir),
    '--conversion-dir', str(CONVERSION_DIR),
    '--qualitative-top-n', str(QUALITATIVE_TOP_N),
])
RESULT_BUNDLE = PROJECT_ROOT / 'downloads' / 'phowhisper_cpp_paper_bundle.zip'
run([
    sys.executable, str(SCRIPT_PATH), 'package-results',
    '--benchmark-dir', str(output_dir),
    '--conversion-dir', str(CONVERSION_DIR),
    '--output-zip', str(RESULT_BUNDLE),
])

print('Benchmark dir:', output_dir)
print('Conversion dir:', CONVERSION_DIR)
print('Bundle:', RESULT_BUNDLE)


In [ ]:
from IPython.display import Image, Markdown, display

evidence_dir = output_dir / 'evidence'
figure_candidates = [
    evidence_dir / 'figures' / 'avg_wer_by_model.png',
    evidence_dir / 'figures' / 'avg_rtf_by_model.png',
    evidence_dir / 'figures' / 'wer_vs_rtf_scatter.png',
    evidence_dir / 'figures' / 'conversion_artifact_sizes.png',
]
display(Markdown((evidence_dir / 'paper_evidence.md').read_text(encoding='utf-8')))
for figure_path in figure_candidates:
    if figure_path.exists():
        display(Image(filename=str(figure_path)))

try:
    from google.colab import files
    print('Use files.download(str(RESULT_BUNDLE)) when you are ready to download the full bundle.')
except Exception:
    print('Not running inside google.colab; download the bundle manually from', RESULT_BUNDLE)
